# Pelagic spin-up validation notebook

This notebook follows the following workflow:

1. **Configuration** for paths, variables, layers, and output folders.
2. **Helper functions** for loading model output and handling grid/time quirks.
3. **Analysis functions** for trends, subdomain diagnostics, observations, and satellite comparison.
4. **Entry-point examples** that can be toggled on when you want to run a specific workflow.

The main cleanup change is that map-based diagnostics now **export individual PNG frames** instead of rendering animations inline in the notebook.


In [ ]:
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import xarray as xr
from IPython.display import display
import os


In [ ]:
# -----------------------------------------------------------------------------
# CONFIGURATION
# -----------------------------------------------------------------------------

POSTPROC_DIR = Path('/export/lv9/projects/dws/results/validation/pelagic/')
BASE_OUTPUT_DIR = Path('/export/lv9/projects/dws/model_output/archived_runs/')

base_dir = "/export/lv9/projects/dws/model_output/archived_runs"
frames_root = "/export/lv9/projects/dws/results/animation/pelagic/frames_all_spinups"
os.makedirs(frames_root, exist_ok=True)

DEFAULT_SPINUP = 'spinup_05'
YEAR = 2015

FRAME_OUTPUT_DIR = (
    POSTPROC_DIR / DEFAULT_SPINUP / f"{YEAR}_spinup_frames_basemap" / variable
)
FRAME_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Map window
MAP_EXTENT_LONLAT = (4.0, 6.6, 52.5, 53.8)

# Transect definition
I_LON = 152
LAT_START = 90
LAT_STOP = 153

# Color scale
VMIN = 0
VMAX = 4000
CMAP = cmocean.cm.matter


# Choose ONE variable
variable = "P1c"
label = "Diatoms (P1c)"

# variable = "P2c"
# label = "Flagellates (P2c)"

# variable = "P3c"
# label = "PicoPhytoPlankton (P3c)"



In [ ]:


spinups = sorted([d for d in os.listdir(base_dir) if d.startswith("spinup_")])
frame_counter = 0

for spinup in spinups:
    print(f"\n=== Processing spinup: {spinup} ===")

    spinup_dir = os.path.join(base_dir, spinup)
    nc_files = sorted(glob.glob(os.path.join(spinup_dir, "dws_500m.3d.*.nc")))

    # Create subfolder for this spinup
    spinup_frame_dir = os.path.join(frames_root, spinup)
    os.makedirs(spinup_frame_dir, exist_ok=True)

    for file_path in nc_files:
        print(f"  → Reading {file_path}")

        with xr.open_dataset(file_path) as ds:

            # Fix curvilinear grid
            lon_da = ds.lonc.ffill("xc").bfill("xc").ffill("yc").bfill("yc")
            lat_da = ds.latc.ffill("xc").bfill("xc").ffill("yc").bfill("yc")
            lon = lon_da.values
            lat = lat_da.values

            # Extract variable
            data_var = ds[variable]
            if "level" in data_var.dims:
                data_var = data_var.isel(level=-1)

            # Mask variable
            mask_da = ds["elev"].squeeze(drop=True)
            mask_time_dim = _find_time_dim(mask_da)
            mask_da = _drop_duplicate_time(mask_da, mask_time_dim)

            # Build transect
            tran_lon = lon[LAT_START:LAT_STOP, I_LON]
            tran_lat = lat[LAT_START:LAT_STOP, I_LON]
            valid = np.isfinite(tran_lon) & np.isfinite(tran_lat)
            tran_lon = tran_lon[valid]
            tran_lat = tran_lat[valid]

            # Loop through time steps
            for t_idx in range(data_var.sizes["time"]):

                frame = data_var.isel(time=t_idx).values.astype(float)

                # Apply mask
                mask2d = mask_da.isel({mask_time_dim: t_idx}).values
                wet_mask = mask2d > -1.1869
                frame[~wet_mask] = np.nan

                timestamp = str(ds.time.values[t_idx])[:10]  # YYYY-MM-DD

                # Plot
                fig = plt.figure(figsize=(8, 6))
                ax = plt.axes(projection=ccrs.Mercator())
                ax.set_extent(MAP_EXTENT_LONLAT, crs=ccrs.PlateCarree())

                # Basemap
                ctx.add_basemap(
                    ax,
                    crs=ax.projection.to_string(),
                    source=ctx.providers.Esri.WorldShadedRelief,
                    zorder=1,
                )
                txt = ax.texts[-1]
                txt.set_position([0.99, 0.02])
                txt.set_ha("right")

                # Field
                mesh = ax.pcolormesh(
                    lon,
                    lat,
                    frame,
                    transform=ccrs.PlateCarree(),
                    cmap=CMAP,
                    vmin=VMIN,
                    vmax=VMAX,
                    shading="gouraud",
                    zorder=2,
                )

                # Transect
                ax.plot(
                    tran_lon,
                    tran_lat,
                    "--",
                    color="black",
                    linewidth=2,
                    transform=ccrs.PlateCarree(),
                    zorder=5,
                )

                # Colorbar
                cbar = plt.colorbar(mesh, ax=ax, shrink=0.9, pad=0.05)
                cbar.set_label("Biomass (mgC m$^{-3}$)", fontsize=12)

                ax.set_title(f"{label} | {spinup} | {timestamp}")

                # Save frame with:
                #   - global counter (for MP4)
                #   - date (for readability)
                frame_path = os.path.join(
                    spinup_frame_dir,
                    f"frame_{frame_counter:06d}_{timestamp}.png"
                )

                plt.savefig(frame_path, dpi=200, bbox_inches="tight")
                plt.close(fig)

                frame_counter += 1


In [ ]:
def export_mp4_from_frames(frame_dir,
                           output_name="animation.mp4",
                           fps=40): 

    frame_dir = Path(frame_dir)
    output_path = Path(output_name)

    frame_files = sorted(frame_dir.rglob("*.png"))
    if not frame_files:
        print(f"No PNG frames found in {frame_dir}")
        return

    print(f"Creating MP4 animation → {output_path}")
    print(f"Found {len(frame_files)} frames")

    writer = animation.FFMpegWriter(
        fps=fps,
        codec="libx264",
        bitrate=1800,
        extra_args=["-pix_fmt", "yuv420p"]   # <-- Windows compatible
    )

    first_img = plt.imread(frame_files[0])
    fig, ax = plt.subplots(figsize=(8, 6))
    im = ax.imshow(first_img)
    ax.axis("off")

    with writer.saving(fig, str(output_path), dpi=200):
        for f in frame_files:
            img = plt.imread(f)
            im.set_data(img)
            writer.grab_frame()

    plt.close(fig)
    print(f"MP4 saved: {output_path}")


In [ ]:
export_mp4_from_frames(
    frames_root,
    output_name=Path(frames_root) / "pelagic_all_spinups.mp4",
    fps=25
)


In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# -----------------------------
# USER SETTINGS
# -----------------------------
variable    = "temp"         # <--- your requested variable
label       = "Temperature (degrees)"

#variable    = "P1c"         # <--- your requested variable
#label       = "Diatoms (P1c)"

# variable = "P2c"
# label = "Flagellates (P2c)"

# variable = "P3c"
# label = "PicoPhytoPlankton (P3c)"

INDEX_LON_TRANSECT = 152
LAT_START = 90
LAT_STOP  = 153

VMIN = 0
VMAX = 150
CMAP = cmocean.cm.matter

# -----------------------------
# PATHS
# -----------------------------
base_dir = "/export/lv9/projects/dws/model_output/archived_runs"

# Root folder for all transect frames
frames_root = Path("/export/lv9/projects/dws/results/animation/pelagic/frames_all_spinups_transect")
frames_root.mkdir(parents=True, exist_ok=True)

spinups = sorted([d for d in os.listdir(base_dir) if d.startswith("spinup_")])

frame_counter = 0

# helper
def _find_xy_dims(da: xr.DataArray, time_dim: str, z_dim: str) -> tuple[str, str]:
    remain = [d for d in da.dims if d not in (time_dim, z_dim)]
    if len(remain) != 2:
        raise ValueError(
            f"Expected 2 horizontal dims after removing time/z, got {remain} from {da.dims}"
        )
    return remain[0], remain[1]
    
# -----------------------------




In [ ]:

frame_counter = 0   # global counter for MP4 ordering

for spinup in spinups:
    print(f"\n=== Processing transect for {spinup} ===")

    spinup_dir = Path(base_dir) / spinup
    nc_files = sorted(spinup_dir.glob("dws_500m.3d.*.nc"))

    # Folder for this spinup
    spinup_frame_dir = frames_root / spinup
    spinup_frame_dir.mkdir(exist_ok=True)

    for file_path in nc_files:
        print(f"  → Reading {file_path}")

        with xr.open_dataset(file_path, decode_times=True) as ds:

            # -------------------------------
            # Extract transect
            # -------------------------------
            pack = extract_transect(
                ds=ds,
                var_name=variable,
                i_lon=INDEX_LON_TRANSECT,
                lat_start=LAT_START,
                lat_stop=LAT_STOP,
            )

            transect = pack["transect"]
            lat_line = np.asarray(pack["lat_line"].values, dtype=float)
            time_dim = pack["time_dim"]
            z_dim = pack["z_dim"]
            y_dim = pack["y_dim"]

            # -------------------------------
            # Elevation + bathymetry
            # -------------------------------
            elev_name = _pick_existing_var(ds, ELEVATION_VAR_CANDIDATES)
            bathy_name = _pick_existing_var(ds, BATHY_VAR_CANDIDATES)

            elev_da = ds[elev_name]
            elev_time_dim = _find_time_dim(elev_da)
            elev_y_dim, elev_x_dim = _find_horizontal_dims(elev_da, exclude=(elev_time_dim,))
            elev_line = elev_da.isel({
                elev_x_dim: INDEX_LON_TRANSECT,
                elev_y_dim: slice(LAT_START, LAT_STOP),
            }).transpose(elev_time_dim, elev_y_dim)

            bathy_da = ds[bathy_name]
            bathy_y_dim, bathy_x_dim = _find_horizontal_dims(bathy_da)
            bathy_line = bathy_da.isel({
                bathy_x_dim: INDEX_LON_TRANSECT,
                bathy_y_dim: slice(LAT_START, LAT_STOP),
            })

            # -------------------------------
            # Build static mask
            # -------------------------------
            bottom_vals = np.where(
                np.asarray(bathy_line.values) <= -9990,
                np.nan,
                -np.asarray(bathy_line.values, dtype=float),
            )

            valid_idx = np.flatnonzero(
                np.isfinite(lat_line) & np.isfinite(bottom_vals)
            )
            if valid_idx.size < 2:
                raise ValueError("Not enough valid transect columns.")

            # Apply static mask
            transect = transect.isel({y_dim: valid_idx})
            elev_line = elev_line.isel({elev_y_dim: valid_idx})
            lat_vals = lat_line[valid_idx]
            bottom_vals = bottom_vals[valid_idx]

            # -------------------------------
            # Vertical edges
            # -------------------------------
            nz = transect.sizes[z_dim]
            sigma_edges = np.linspace(0, 1, nz + 1)[:, None]

            ymin = float(np.nanmin(bottom_vals)) - 0.5
            ymax = 1.4

            # -------------------------------
            # Loop through time frames
            # -------------------------------
            times = transect[time_dim]
            frame_indices = np.arange(0, transect.sizes[time_dim], 1)

            for idx in frame_indices:

                temp_vals = np.asarray(
                    transect.isel({time_dim: idx})
                    .transpose(z_dim, y_dim)
                    .values,
                    dtype=float,
                )
                surface_vals = np.asarray(
                    elev_line.isel({elev_time_dim: idx}).values,
                    dtype=float,
                )

                # Wet mask
                valid_frame = (
                    np.isfinite(surface_vals)
                    & np.isfinite(bottom_vals)
                    & (surface_vals > bottom_vals)
                )

                # -------------------------------
                # Plot
                # -------------------------------
                fig, ax = plt.subplots(figsize=(10, 5))

                if valid_frame.sum() >= 2:
                    lat_plot = lat_vals[valid_frame]
                    bottom_plot = bottom_vals[valid_frame]
                    surface_plot = surface_vals[valid_frame]
                    temp_plot = np.ma.masked_invalid(temp_vals[:, valid_frame])

                    x_edges = _compute_edges(lat_plot)
                    x_edges_2d = np.broadcast_to(
                        x_edges[None, :], (temp_plot.shape[0] + 1, x_edges.size)
                    )
                    z_edges_2d = _make_vertical_edges(surface_plot, bottom_plot)

                    ax.pcolormesh(
                        x_edges_2d,
                        z_edges_2d,
                        temp_plot,
                        shading="auto",
                        cmap=CMAP,
                        vmin=VMIN,
                        vmax=VMAX,
                    )

                # Seafloor
                ax.fill_between(
                    lat_vals,
                    bottom_vals,
                    ymin,
                    color="#dcd5c1",
                    alpha=1,
                )

                # Surface line
                if valid_frame.sum() >= 2:
                    ax.plot(lat_plot, surface_plot, color="white", linewidth=1.2)

                # Bathymetry line
                ax.plot(lat_vals, bottom_vals, color="black", linewidth=1.8)

                timestamp = str(times.values[idx])[:10]
                ax.set_title(f"{label} transect | {timestamp}")
                ax.set_xlabel("Latitude")
                ax.set_ylabel("Depth (m)")
                ax.set_ylim(ymin, ymax)
                ax.grid(True, alpha=0.25)

                # -------------------------------
                # Save frame
                # -------------------------------
                frame_path = spinup_frame_dir / f"frame_{frame_counter:06d}_{timestamp}.png"
                fig.savefig(frame_path, dpi=200, bbox_inches="tight")
                plt.close(fig)

                frame_counter += 1


In [ ]:
# change the file name
export_mp4_from_frames(
    "/export/lv9/projects/dws/results/animation/pelagic/frames_all_spinups_transect",
    output_name="/export/lv9/projects/dws/results/animation/pelagic/pelagic_all_spinups_transect.mp4",
    fps=40
)